In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                           CONSOLIDAÇÃO DOS DADOS                             ║
# ║        Unificação, Diagnóstico e Geração de Base Final por Submercado         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Versão      : 3.5 (Diagnóstico de Maio integrado)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de consolidação (Step 3) da arquitetura de
#  dados do TCC. Ele une, por submercado (NORTE, NORDESTE, SUL, SUDESTE), os
#  19 datasets já coletados e agrupados nas etapas anteriores, produzindo uma
#  base única por região em formato Parquet, pronta para modelagem.
#
#  Além da consolidação, o script executa um diagnóstico automático que
#  rastreia, passo a passo, se os dados do período mais recente (mês/ano
#  alvo configurável) sobrevivem a cada merge incremental — útil para
#  identificar em qual dataset um período específico se perde no pipeline.
#
#  Ao final da execução, o script gera:
#    (a) um arquivo Parquet consolidado por submercado (compressão Snappy);
#    (b) log de diagnóstico no terminal (via Rich) por região;
#    (c) um relatório PDF com estatísticas de consolidação e log detalhado.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script não requer entradas manuais do usuário. Os parâmetros de controle
#  estão definidos nas constantes abaixo (seção "CONFIGURAÇÕES"):
#
#    ANO_ALVO / MES_ALVO : período usado no diagnóstico de rastreamento
#                          (padrão: 2026 / maio)
#    DIR_RAIZ            : caminho raiz de armazenamento
#                          → Google Drive (/content/drive/MyDrive/TCC_PLD_Project)
#                          → local (./TCC_DATA_MASTER) se Drive não estiver montado
#
#  Arquivos de entrada esperados (gerados pela etapa anterior de agrupamento):
#    DIR_RAIZ / "2-DADOS" / "2-DADOS-AGRUPADOS" / <subpasta_dataset> /
#        <PREFIXO>_<REGIAO>_<SUFIXO>.csv
#    Exemplos: 01_CCEE_NORDESTE_PLD.csv | 17_INMET_SUL_METEOROLOGIA.csv
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Parquet consolidado →  DIR_RAIZ / "2-DADOS" / "3-DADOS-CONSOLIDADOS" /
#     DADOS_CONSOLIDADOS_<REGIAO>.parquet   (compressão Snappy)
#
#  2. Relatório PDF →  DIR_RAIZ / "8-RELATÓRIOS" /
#     Relatorio_Consolidacao_<DD-MM-AAAA_HH-MM-SS>.pdf
#     Contém: resumo das dimensões finais, distribuição por fonte de dados
#     e log detalhado por dataset/região.
#
#  3. Saída no terminal (Rich):
#     • Log de consolidação por dataset (BASE / MERGED / AUSENTE / ERRO)
#     • Diagnóstico de rastreamento do período alvo (3 passos, ver abaixo)
#     • Tabela resumo com estatísticas finais por submercado
#     • Aviso específico quando um submercado tem datasets ausentes
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DATASETS CONSOLIDADOS (resumo)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  N°  Label               Origem  Tipo                    Descrição
#  01  01-PLD              CCEE    target                  Preço de Liquidação das Diferenças
#  02  02-INTERCAMBIO      ONS     feature_operacional      Intercâmbio Nacional
#  03  03-BALANCO          ONS     feature_operacional      Balanço de Energia
#  04  04-CMO              ONS     target                  Custo Marginal de Operação
#  05  05-CURVA            ONS     feature_operacional      Curva de Carga
#  06  06-EAR              ONS     feature_operacional      Energia Armazenada
#  07  07-ENA              ONS     feature_operacional      Energia Natural Afluente
#  08  08-CARGA            ONS     feature_operacional      Carga de Energia
#  09  09-CVU              CCEE    target                  Custo Variável Unitário
#  10  10-VOL-ESPERA       ONS     feature_operacional      Volume de Espera (ausente p/ NORTE)
#  11  11-HIDRO            ONS     feature_operacional      Dados Hidrológicos
#  12  12-DISPONIBILIDADE  ONS     feature_operacional      Disponibilidade de Usinas
#  13  13-FAT-CAPAC        ONS     feature_operacional      Fator de Capacidade
#  14  14-GERACAO          ONS     feature_operacional      Geração por Usina
#  15  15-VERTIDA          ONS     feature_operacional      Energia Vertida
#  16  16-TERMICA          ONS     feature_operacional      Geração Térmica
#  17  17-METEOROLOGIA     INMET   feature_meteorologica    Variáveis meteorológicas
#  18  18-CARGA-VERIF      ONS     feature_operacional      Carga Verificada (API)
#  19  19-DOLAR            BCB     feature_financeira       Cotação PTAX do dólar
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de DIR_RAIZ.
#     Depende dos arquivos já gerados pelas etapas anteriores de coleta e
#     agrupamento (Steps 1 e 2 da arquitetura de dados do TCC).
#
#  2. RESOLUÇÃO DE CAMINHOS DOS ARQUIVOS
#     montar_caminho_direto() constrói o caminho exato do CSV a partir do
#     padrão de nomenclatura conhecido (PREFIXO_REGIAO_SUFIXO.csv), evitando
#     percorrer todo o sistema de arquivos. Caso o caminho direto não seja
#     encontrado, resolver_caminho() aciona encontrar_arquivo_fallback(),
#     que faz uma busca recursiva (os.walk) como último recurso.
#
#  3. NORMALIZAÇÃO DE CHAVES APÓS MERGE OUTER
#     Merges do tipo 'outer' podem converter colunas inteiras (KEY_ANO,
#     KEY_MES, KEY_DIA, KEY_HORA) para float quando surgem valores ausentes.
#     normalizar_chaves() reconverte essas colunas para Int64 (nullable)
#     após cada merge, evitando falhas silenciosas em filtros e ordenações.
#
#  4. TRATAMENTO DE DUPLICATAS
#     checar_duplicatas() remove duplicatas por chave de merge, mantendo o
#     primeiro registro (keep='first'). O diagnóstico de rastreamento avisa
#     explicitamente quando há duplicatas no período alvo antes da remoção,
#     já que isso pode reduzir a contagem de linhas do período de forma
#     inesperada.
#
#  5. DIAGNÓSTICO DE RASTREAMENTO DE PERÍODO
#     diagnosticar_maio() executa três passos para cada submercado:
#       Passo 1 — verifica a presença do período alvo em cada CSV de origem;
#       Passo 2 — simula o merge incremental e aponta em qual dataset o
#                 período alvo desaparece ou é reduzido;
#       Passo 3 — inspeciona o Parquet final gerado por processar_regiao()
#                 e confirma se o período alvo está presente na base
#                 consolidada.
#     O nome da função e o período padrão (maio/2026) refletem o caso de uso
#     original, mas o período pode ser alterado via ANO_ALVO / MES_ALVO.
#
#  6. AUSÊNCIA DE DATASETS POR SUBMERCADO
#     Nem todo dataset existe para todos os submercados — o caso mais notável
#     é o dataset 10-VOL-ESPERA, que não é publicado para o submercado NORTE.
#     Esses casos são tratados como AUSENTE no log e sinalizados ao final da
#     execução, sem interromper o processamento das demais regiões.
#
#  7. GESTÃO DE MEMÓRIA
#     otimizar_memoria() faz downcast de inteiros e floats e converte
#     KEY_SUBMERCADO para category. Cada DataFrame temporário é liberado
#     com gc.collect() após o merge, mantendo o uso de memória sob controle
#     ao longo dos 19 datasets por região.
#
#  8. REPRODUTIBILIDADE
#     O relatório PDF inclui timestamp de geração no nome do arquivo, e o
#     log detalhado registra fonte, tipo e status de cada dataset por
#     região, garantindo rastreabilidade completa da consolidação.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas auxiliares
#  pandas            1.5            Manipulação de DataFrames, merges e datas
#  pyarrow           10.0           Leitura/escrita de arquivos Parquet
#  rich              13.0           Tabelas, painéis e progresso no terminal
#  reportlab         3.6            Geração do relatório em PDF
#
#  Instalação rápida (Google Colab / pip):
#      !pip install rich reportlab pyarrow
#  (numpy e pandas já estão disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS UTILIZADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TCC_PLD_Project/
#  ├── 2-DADOS/
#  │   ├── 2-DADOS-AGRUPADOS/          (entrada desta etapa)
#  │   │   ├── 01-PLD/                 01_CCEE_<REGIAO>_PLD.csv
#  │   │   ├── 02-INTERCAMBIO/         02_ONS_<REGIAO>_INTERCAMBIO.csv
#  │   │   ├── ...
#  │   │   └── 19-DOLAR/               19_BCB_<REGIAO>_DOLAR.csv
#  │   └── 3-DADOS-CONSOLIDADOS/        (saída desta etapa)
#  │       ├── DADOS_CONSOLIDADOS_NORTE.parquet
#  │       ├── DADOS_CONSOLIDADOS_NORDESTE.parquet
#  │       ├── DADOS_CONSOLIDADOS_SUL.parquet
#  │       └── DADOS_CONSOLIDADOS_SUDESTE.parquet
#  └── 8-RELATÓRIOS/
#      └── Relatorio_Consolidacao_<DD-MM-AAAA_HH-MM-SS>.pdf
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install rich reportlab pyarrow
#
#  Célula 3 — Executar o script:
#      exec(open('consolidacao_dados_tcc_v3.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os, gc, time, warnings
from datetime import datetime

try:
    from rich.console import Console
    from rich.table import Table as RichTable
    from rich.panel import Panel
    from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn
    from rich.theme import Theme
    from rich import box
except ImportError:
    os.system("pip install rich reportlab pyarrow")
    from rich.console import Console
    from rich.table import Table as RichTable
    from rich.panel import Panel
    from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn
    from rich.theme import Theme
    from rich import box

from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                 Table as PDFTable, TableStyle, HRFlowable)
from reportlab.lib.enums import TA_CENTER

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

custom_theme = Theme({
    "info": "dim cyan", "warning": "bold yellow",
    "error": "bold red", "success": "bold green",
    "header": "bold white on blue",
})
console = Console(theme=custom_theme)

# ==============================================================================
# ⚙️ CONFIGURAÇÕES DO AMBIENTE
# ==============================================================================

DIR_RAIZ = r"/content/drive/MyDrive/TCC_PLD_Project"
if not os.path.exists(DIR_RAIZ):
    DIR_RAIZ = "./TCC_DATA_MASTER"

DIR_ENTRADA   = os.path.join(DIR_RAIZ, "2-DADOS/2-DADOS-AGRUPADOS")
DIR_SAIDA     = os.path.join(DIR_RAIZ, "2-DADOS/3-DADOS-CONSOLIDADOS")
DIR_RELATORIO = os.path.join(DIR_RAIZ, "8-RELATÓRIOS")

KEYS_MERGE = ['KEY_SUBMERCADO', 'KEY_ANO', 'KEY_MES', 'KEY_DIA', 'KEY_HORA']

# ==============================================================================
# 📋 CATÁLOGO DE DATASETS (DATASETS_MAP)
#
# Estrutura de cada entrada:
#   "LABEL": (
#       "SUBPASTA",          ← nome da pasta dentro de DIR_ENTRADA
#       "PREFIXO_ARQUIVO",   ← parte fixa antes de _{REGIAO}_ no nome do CSV
#       "SUFIXO_ARQUIVO",    ← parte fixa depois de _{REGIAO}_ no nome do CSV
#       "FONTE",             ← ONS | INMET | CCEE | BCB
#       "TIPO",              ← papel no modelo
#   )
#
# Nome final do arquivo montado em montar_caminho_direto():
#   {PREFIXO_ARQUIVO}_{REGIAO}_{SUFIXO_ARQUIVO}.csv
#
# Exemplos reais dos CSVs:
#   01_CCEE_NORDESTE_PLD.csv
#   02_ONS_NORTE_INTERCAMBIO.csv
#   17_INMET_SUL_METEOROLOGIA.csv
# ==============================================================================

DATASETS_MAP = {
    # LABEL              SUBPASTA                  PREFIXO     SUFIXO                  FONTE   TIPO
    "01-PLD":            ("01-PLD",                "01_CCEE",  "PLD",                  "CCEE",  "target"),
    "02-INTERCAMBIO":    ("02-INTERCAMBIO",         "02_ONS",   "INTERCAMBIO",          "ONS",   "feature_operacional"),
    "03-BALANCO":        ("03-BALANCO-ENERGIA",     "03_ONS",   "BALANCO-ENERGIA",      "ONS",   "feature_operacional"),
    "04-CMO":            ("04-CMO",                 "04_ONS",   "CMO",                  "ONS",   "target"),
    "05-CURVA":          ("05-CURVA",               "05_ONS",   "CURVA-CARGA",          "ONS",   "feature_operacional"),
    "06-EAR":            ("06-EAR",                 "06_ONS",   "EAR",                  "ONS",   "feature_operacional"),
    "07-ENA":            ("07-ENA",                 "07_ONS",   "ENA",                  "ONS",   "feature_operacional"),
    "08-CARGA":          ("08-CARGA",               "08_ONS",   "CARGA",                "ONS",   "feature_operacional"),
    "09-CVU":            ("09-CVU",                 "09_ONS",   "CVU",                  "CCEE",  "target"),
    "10-VOL-ESPERA":     ("10-VOLUME-ESPERA",       "10_ONS",   "VOLUME-ESPERA",        "ONS",   "feature_operacional"),
    "11-HIDRO":          ("11-HIDRO",               "11_ONS",   "HIDRO-RESERVATORIOS",  "ONS",   "feature_operacional"),
    "12-DISPONIBILIDADE":("12-DISPONIBILIDADE",     "12_ONS",   "DISPONIBILIDADE",      "ONS",   "feature_operacional"),
    "13-FAT-CAPAC":      ("13-FATOR-CAPACIDADE",    "13_ONS",   "FATOR-CAPACIDADE",     "ONS",   "feature_operacional"),
    "14-GERACAO":        ("14-GERACAO",             "14_ONS",   "GERACAO-USINA",        "ONS",   "feature_operacional"),
    "15-VERTIDA":        ("15-VERTIDA",             "15_ONS",   "ENERGIA-VERTIDA",      "ONS",   "feature_operacional"),
    "16-TERMICA":        ("16-TERMICA",             "16_ONS",   "GERACAO-TERMICA",      "ONS",   "feature_operacional"),
    "17-METEOROLOGIA":   ("17-METEOROLOGIA",        "17_INMET", "METEOROLOGIA",         "INMET", "feature_meteorologica"),
    "18-CARGA-VERIF":    ("18-CARGA-VERIFICADA",    "18_ONS",   "CARGA-VERIFICADA",     "ONS",   "feature_operacional"),
    "19-DOLAR":          ("19-DOLAR",               "19_BCB",   "DOLAR",                "BCB",   "feature_financeira"),
}

# Dataset 10-VOLUME-ESPERA não existe para NORTE (ausente no catálogo real).
# O pipeline já trata arquivos não encontrados como AUSENTE; esta constante
# serve apenas para documentação e avisos específicos.
DATASETS_AUSENTES_NORTE = {"10-VOL-ESPERA"}

REGIOES = ["NORTE", "NORDESTE", "SUL", "SUDESTE"]
LOGS_PDF = []
STATS_EXECUCAO = []

# Mês/ano alvo do diagnóstico de rastreamento
ANO_ALVO = 2026
MES_ALVO = 5

# ==============================================================================
# 🛠️ FUNÇÕES UTILITÁRIAS
# ==============================================================================

def montar_caminho_direto(subpasta, prefixo, sufixo, regiao):
    """
    Constrói o caminho exato do CSV sem percorrer o sistema de arquivos.

    Padrão real dos arquivos:
        {DIR_ENTRADA}/{SUBPASTA}/{PREFIXO}_{REGIAO}_{SUFIXO}.csv

    Exemplo:
        DIR_ENTRADA/01-PLD/01_CCEE_NORDESTE_PLD.csv
        DIR_ENTRADA/17-METEOROLOGIA/17_INMET_SUL_METEOROLOGIA.csv
    """
    nome_arquivo = f"{prefixo}_{regiao}_{sufixo}.csv"
    caminho = os.path.join(DIR_ENTRADA, subpasta, nome_arquivo)
    return caminho if os.path.exists(caminho) else None


def encontrar_arquivo_fallback(nome_arquivo, diretorio_base):
    """
    Busca recursiva (fallback) caso montar_caminho_direto falhe.
    Útil se a estrutura de pastas mudar inesperadamente.
    """
    for root, _, files in os.walk(diretorio_base):
        if nome_arquivo in files:
            return os.path.join(root, nome_arquivo)
    return None


def otimizar_memoria(df):
    if df.empty:
        return df
    ints   = df.select_dtypes(include=['int64', 'int32']).columns
    df[ints] = df[ints].apply(pd.to_numeric, downcast='integer')
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].apply(pd.to_numeric, downcast='float')
    if 'KEY_SUBMERCADO' in df.columns:
        df['KEY_SUBMERCADO'] = df['KEY_SUBMERCADO'].astype('category')
    return df


def checar_duplicatas(df, nome_dataset):
    """
    Remove duplicatas mantendo keep='first'.
    ATENÇÃO: linhas duplicadas no período alvo serão eliminadas aqui se o
    registro mais antigo (posição mais alta no CSV) vier de outro período —
    o diagnóstico detecta esse caso antes da remoção.
    """
    duplicadas = df.duplicated(subset=KEYS_MERGE, keep=False).sum()
    if duplicadas > 0:
        df = df.drop_duplicates(subset=KEYS_MERGE, keep='first')
    return df


def registrar_log(regiao, dataset, status, obs="", fonte="", tipo=""):
    if status not in ["BASE", "MERGED"]:
        cor = "red" if status in ["ERRO", "AUSENTE"] else "yellow"
        console.print(f"   [{cor}]• {dataset}: {status} ({obs})[/]")
    LOGS_PDF.append([regiao, dataset, fonte, tipo, status, obs])


def ler_csv_robusto(caminho):
    """Tenta ler CSV com separador vírgula e, se falhar, com ponto-e-vírgula."""
    try:
        return pd.read_csv(caminho)
    except Exception:
        return pd.read_csv(caminho, sep=';')


def normalizar_chaves(df):
    """
    Garante que KEY_ANO e KEY_MES sejam inteiros após merges
    (merges outer podem converter int → float quando há NaN).
    Isso evita que filtros como KEY_ANO == 2026 falhem silenciosamente.
    """
    for col in ['KEY_ANO', 'KEY_MES', 'KEY_DIA', 'KEY_HORA']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            # Converte para Int64 (nullable integer) para preservar NaN sem virar float
            df[col] = df[col].astype('Int64')
    return df


def resolver_caminho(nome_log, subpasta, prefixo, sufixo, regiao):
    """
    Tenta o caminho direto primeiro; cai no os.walk como último recurso.
    Retorna None se o arquivo não existir.
    """
    caminho = montar_caminho_direto(subpasta, prefixo, sufixo, regiao)
    if caminho:
        return caminho

    # Fallback: busca pelo nome do arquivo em toda a árvore DIR_ENTRADA
    nome_arquivo = f"{prefixo}_{regiao}_{sufixo}.csv"
    caminho_fb = encontrar_arquivo_fallback(nome_arquivo, DIR_ENTRADA)
    if caminho_fb:
        console.print(
            f"   [dim yellow]⚠ {nome_log}: caminho direto não encontrado, "
            f"usando fallback → {caminho_fb}[/]"
        )
    return caminho_fb

# ==============================================================================
# 🔍 DIAGNÓSTICO: RASTREAMENTO DO PERÍODO ALVO POR REGIÃO
# ==============================================================================

def diagnosticar_maio(regiao):
    """
    Executa três passos de diagnóstico para identificar onde os dados do
    período alvo (ANO_ALVO/MES_ALVO) somem no pipeline de consolidação:

      Passo 1 — Verifica cada CSV de origem individualmente.
      Passo 2 — Simula o merge incremental e detecta em qual dataset o
                período alvo desaparece.
      Passo 3 — Inspeciona o parquet final gerado por processar_regiao().
    """
    console.print(Panel.fit(
        f"[bold magenta]🔍 DIAGNÓSTICO DE {MES_ALVO:02d}/{ANO_ALVO} — {regiao}[/]",
        border_style="magenta"
    ))

    # ── PASSO 1: CSVs de origem ───────────────────────────────────────────────
    console.print("\n[cyan bold][ PASSO 1 ] Verificando presença do período alvo em cada CSV de origem:[/]")

    datasets_com_maio = []
    datasets_sem_maio = []
    datasets_ausentes = []

    for nome_log, (subpasta, prefixo, sufixo, fonte, tipo) in DATASETS_MAP.items():
        caminho = resolver_caminho(nome_log, subpasta, prefixo, sufixo, regiao)

        if not caminho:
            datasets_ausentes.append(nome_log)
            console.print(f"   [dim]— {nome_log}: arquivo não encontrado[/]")
            continue

        try:
            df_tmp = ler_csv_robusto(caminho)

            tem_chaves = all(k in df_tmp.columns for k in ['KEY_ANO', 'KEY_MES'])
            if not tem_chaves:
                console.print(
                    f"   [yellow]⚠ {nome_log}: sem KEY_ANO/KEY_MES "
                    f"— colunas disponíveis: {list(df_tmp.columns[:8])}[/]"
                )
                del df_tmp
                continue

            # Força conversão numérica para evitar comparação str vs int
            df_tmp['KEY_ANO'] = pd.to_numeric(df_tmp['KEY_ANO'], errors='coerce')
            df_tmp['KEY_MES'] = pd.to_numeric(df_tmp['KEY_MES'], errors='coerce')

            n_maio = ((df_tmp['KEY_ANO'] == ANO_ALVO) &
                      (df_tmp['KEY_MES'] == MES_ALVO)).sum()

            # Verifica duplicatas especificamente no período alvo
            dups_maio = 0
            chaves_presentes = [k for k in KEYS_MERGE if k in df_tmp.columns]
            if len(chaves_presentes) == len(KEYS_MERGE):
                mask_maio = ((df_tmp['KEY_ANO'] == ANO_ALVO) &
                             (df_tmp['KEY_MES'] == MES_ALVO))
                df_maio_tmp = df_tmp[mask_maio]
                dups_maio = df_maio_tmp.duplicated(subset=KEYS_MERGE, keep=False).sum()

            if n_maio > 0:
                datasets_com_maio.append(nome_log)
                aviso_dup = (f" [red]⚠ {dups_maio} DUPLICATAS no período alvo "
                             f"(serão removidas!)[/]") if dups_maio > 0 else ""
                console.print(
                    f"   [green]✅ {nome_log}: {n_maio} linhas em "
                    f"{MES_ALVO:02d}/{ANO_ALVO}{aviso_dup}[/]"
                )
            else:
                ano_max = df_tmp['KEY_ANO'].max()
                mes_max = df_tmp[df_tmp['KEY_ANO'] == ano_max]['KEY_MES'].max() \
                          if not pd.isna(ano_max) else "?"
                datasets_sem_maio.append(nome_log)
                console.print(
                    f"   [yellow]⚠ {nome_log}: SEM dados {MES_ALVO:02d}/{ANO_ALVO} "
                    f"— último registro: {int(mes_max):02d}/{int(ano_max)}[/]"
                )

            del df_tmp
            gc.collect()

        except Exception as e:
            console.print(f"   [red]❌ {nome_log}: erro ao ler — {e}[/]")

    console.print(
        f"\n   [bold]Resumo CSVs:[/] {len(datasets_com_maio)} com o período alvo "
        f"| {len(datasets_sem_maio)} sem o período alvo "
        f"| {len(datasets_ausentes)} ausentes"
    )

    if not datasets_com_maio:
        console.print(
            "[bold red]   🚨 NENHUM CSV de origem possui dados do período alvo. "
            "O problema está no ETL anterior (Step 1/2), não aqui.[/]"
        )
        return

    # ── PASSO 2: Simulação incremental do merge ───────────────────────────────
    console.print(
        "\n[cyan bold][ PASSO 2 ] Simulando merge incremental "
        "e rastreando quando o período alvo desaparece:[/]"
    )

    df_sim = None
    maio_referencia = 0

    for nome_log, (subpasta, prefixo, sufixo, fonte, tipo) in DATASETS_MAP.items():
        caminho = resolver_caminho(nome_log, subpasta, prefixo, sufixo, regiao)
        if not caminho:
            continue

        try:
            df_tmp = ler_csv_robusto(caminho)

            if 'KEY_SUBMERCADO' in df_tmp.columns:
                df_tmp['KEY_SUBMERCADO'] = (df_tmp['KEY_SUBMERCADO']
                                            .astype(str).str.upper())

            df_tmp = otimizar_memoria(df_tmp)

            # — Aviso de duplicatas no período alvo antes de remover —
            chaves_ok = all(k in df_tmp.columns for k in KEYS_MERGE)
            if chaves_ok:
                for col in ['KEY_ANO', 'KEY_MES']:
                    df_tmp[col] = pd.to_numeric(df_tmp[col], errors='coerce')
                mask_maio = ((df_tmp['KEY_ANO'] == ANO_ALVO) &
                             (df_tmp['KEY_MES'] == MES_ALVO))
                dups_maio_antes = df_tmp[mask_maio].duplicated(
                    subset=KEYS_MERGE, keep=False).sum()
                if dups_maio_antes > 0:
                    console.print(
                        f"   [bold red]🚨 {nome_log}: {dups_maio_antes} linhas "
                        f"DUPLICADAS em {MES_ALVO:02d}/{ANO_ALVO} — drop_duplicates vai "
                        f"eliminar metade delas![/]"
                    )

            df_tmp = checar_duplicatas(df_tmp, nome_log)
            df_tmp = normalizar_chaves(df_tmp)

            if df_sim is None:
                df_sim = df_tmp
                if all(k in df_sim.columns for k in ['KEY_ANO', 'KEY_MES']):
                    maio_referencia = int(((df_sim['KEY_ANO'] == ANO_ALVO) &
                                           (df_sim['KEY_MES'] == MES_ALVO)).sum())
                console.print(
                    f"   [blue]BASE → {nome_log}: {maio_referencia} linhas "
                    f"do período alvo no dataset inicial[/]"
                )
            else:
                maio_antes_merge = 0
                if all(k in df_sim.columns for k in ['KEY_ANO', 'KEY_MES']):
                    maio_antes_merge = int(((df_sim['KEY_ANO'] == ANO_ALVO) &
                                            (df_sim['KEY_MES'] == MES_ALVO)).sum())

                df_sim = pd.merge(df_sim, df_tmp, on=KEYS_MERGE, how='outer')
                df_sim = normalizar_chaves(df_sim)

                maio_depois_merge = 0
                if all(k in df_sim.columns for k in ['KEY_ANO', 'KEY_MES']):
                    maio_depois_merge = int(((df_sim['KEY_ANO'] == ANO_ALVO) &
                                             (df_sim['KEY_MES'] == MES_ALVO)).sum())

                delta = maio_depois_merge - maio_antes_merge

                if maio_antes_merge > 0 and maio_depois_merge == 0:
                    console.print(
                        f"   [bold red]🚨 PERÍODO ALVO ZEROU após merge com {nome_log}! "
                        f"{maio_antes_merge} → 0 linhas[/]"
                    )
                elif delta < 0:
                    console.print(
                        f"   [yellow]⚠ {nome_log}: período alvo reduziu "
                        f"{maio_antes_merge} → {maio_depois_merge} "
                        f"({delta:+d} linhas)[/]"
                    )
                elif delta > 0:
                    console.print(
                        f"   [green]+ {nome_log}: período alvo cresceu "
                        f"{maio_antes_merge} → {maio_depois_merge} "
                        f"({delta:+d} linhas novas)[/]"
                    )
                # else: sem mudança, silêncio para não poluir o log

            del df_tmp
            gc.collect()

        except Exception as e:
            console.print(f"   [red]❌ Erro no merge de diagnóstico {nome_log}: {e}[/]")

    if df_sim is not None:
        maio_final_sim = 0
        if all(k in df_sim.columns for k in ['KEY_ANO', 'KEY_MES']):
            maio_final_sim = int(((df_sim['KEY_ANO'] == ANO_ALVO) &
                                   (df_sim['KEY_MES'] == MES_ALVO)).sum())
        console.print(
            f"\n   [bold]Resultado da simulação:[/] "
            f"{maio_final_sim} linhas do período alvo no df_master simulado"
        )
        del df_sim
        gc.collect()

    # ── PASSO 3: Inspeção do parquet final ────────────────────────────────────
    console.print("\n[cyan bold][ PASSO 3 ] Inspecionando parquet final gerado:[/]")

    caminho_parquet = os.path.join(
        DIR_SAIDA, f"DADOS_CONSOLIDADOS_{regiao}.parquet"
    )

    if not os.path.exists(caminho_parquet):
        console.print(f"   [red]Parquet não encontrado: {caminho_parquet}[/]")
        return

    df_final = pd.read_parquet(caminho_parquet)

    if 'KEY_ANO' not in df_final.columns or 'KEY_MES' not in df_final.columns:
        console.print("   [red]Parquet sem KEY_ANO/KEY_MES![/]")
        del df_final
        return

    # Tipos das chaves — problema clássico: int → float após outer merge
    console.print(
        f"   Tipos: KEY_ANO={df_final['KEY_ANO'].dtype}, "
        f"KEY_MES={df_final['KEY_MES'].dtype}, "
        f"KEY_DIA={df_final.get('KEY_DIA', pd.Series(dtype='object')).dtype}"
    )
    console.print(
        f"   Anos únicos no parquet: "
        f"{sorted(df_final['KEY_ANO'].dropna().unique().tolist())}"
    )

    # Distribuição temporal (últimos 8 períodos)
    console.print("\n   Distribuição por ano/mês (últimos 8 períodos no parquet):")
    dist = (
        df_final
        .groupby(['KEY_ANO', 'KEY_MES'], dropna=False)
        .size()
        .reset_index(name='Linhas')
        .sort_values(['KEY_ANO', 'KEY_MES'])
    )
    console.print(dist.tail(8).to_string(index=False))

    # Verifica o período alvo especificamente (com conversão segura)
    df_final['KEY_ANO'] = pd.to_numeric(df_final['KEY_ANO'], errors='coerce')
    df_final['KEY_MES'] = pd.to_numeric(df_final['KEY_MES'], errors='coerce')
    mask_maio = (df_final['KEY_ANO'] == ANO_ALVO) & (df_final['KEY_MES'] == MES_ALVO)
    n_maio_parquet = int(mask_maio.sum())

    if n_maio_parquet > 0:
        console.print(
            f"\n   [bold green]✅ Parquet contém {n_maio_parquet} linhas "
            f"de {MES_ALVO:02d}/{ANO_ALVO}[/]"
        )
    else:
        ano_max = df_final['KEY_ANO'].max()
        mes_max = (df_final[df_final['KEY_ANO'] == ano_max]['KEY_MES'].max()
                   if not pd.isna(ano_max) else "?")
        console.print(
            f"\n   [bold red]❌ Parquet NÃO contém dados de "
            f"{MES_ALVO:02d}/{ANO_ALVO}[/]"
        )
        console.print(
            f"   Último registro encontrado: "
            f"{int(mes_max):02d}/{int(ano_max)}"
        )

    del df_final
    gc.collect()

# ==============================================================================
# 🔄 MOTOR DE CONSOLIDAÇÃO
# ==============================================================================

def processar_regiao(regiao):
    start_time = time.time()
    console.print(f"\n[bold blue]🌍 PROCESSANDO REGIÃO: {regiao}[/]")

    df_master = None
    arquivos_processados = 0
    ausentes = []

    contadores = {
        "fonte": {"ONS": 0, "INMET": 0, "CCEE": 0, "BCB": 0},
        "tipo":  {"feature_operacional": 0, "feature_meteorologica": 0,
                  "target": 0, "feature_financeira": 0}
    }
    cols_por_ds = {}

    with Progress(SpinnerColumn(), TextColumn("{task.description}"),
                  BarColumn(), transient=True) as progress:
        task = progress.add_task(f"Consolidando {regiao}...", total=len(DATASETS_MAP))

        for nome_log, (subpasta, prefixo, sufixo, fonte, tipo) in DATASETS_MAP.items():
            caminho = resolver_caminho(nome_log, subpasta, prefixo, sufixo, regiao)

            if not caminho:
                ausentes.append(nome_log)
                registrar_log(regiao, nome_log, "AUSENTE",
                              "Arquivo não encontrado", fonte, tipo)
                progress.advance(task)
                continue

            try:
                df_temp = ler_csv_robusto(caminho)

                if 'KEY_SUBMERCADO' in df_temp.columns:
                    df_temp['KEY_SUBMERCADO'] = (df_temp['KEY_SUBMERCADO']
                                                 .astype(str).str.upper())

                df_temp = otimizar_memoria(df_temp)
                df_temp = checar_duplicatas(df_temp, nome_log)

                n_cols_dados = len([c for c in df_temp.columns
                                    if c not in KEYS_MERGE])

                if df_master is None:
                    df_master = df_temp
                    # Normaliza chaves já na base
                    df_master = normalizar_chaves(df_master)
                    cols_por_ds[nome_log] = n_cols_dados
                    registrar_log(regiao, nome_log, "BASE",
                                  f"Início: {len(df_temp)} linhas", fonte, tipo)
                else:
                    cols_antes = len(df_master.columns)
                    df_master = pd.merge(df_master, df_temp,
                                         on=KEYS_MERGE, how='outer')
                    # ⚠️ Normaliza chaves após cada merge outer para evitar
                    # que NaNs convertam KEY_ANO/MES para float e quebrem
                    # filtros e ordenações subsequentes.
                    df_master = normalizar_chaves(df_master)
                    cols_novas = len(df_master.columns) - cols_antes
                    cols_por_ds[nome_log] = cols_novas
                    registrar_log(regiao, nome_log, "MERGED",
                                  f"+{cols_novas} features", fonte, tipo)

                contadores["fonte"][fonte] = contadores["fonte"].get(fonte, 0) + 1
                contadores["tipo"][tipo]   = contadores["tipo"].get(tipo, 0) + 1
                arquivos_processados += 1

                del df_temp
                gc.collect()

            except Exception as e:
                registrar_log(regiao, nome_log, "ERRO", str(e), fonte, tipo)

            progress.advance(task)

    # ── Pós-processamento ─────────────────────────────────────────────────────
    tempo_total = time.time() - start_time

    if df_master is not None and not df_master.empty:
        chaves_ord = {'KEY_ANO', 'KEY_MES', 'KEY_DIA', 'KEY_HORA'}
        if chaves_ord.issubset(df_master.columns):
            df_master = df_master.sort_values(
                by=['KEY_ANO', 'KEY_MES', 'KEY_DIA', 'KEY_HORA'])

        nome_saida    = f"DADOS_CONSOLIDADOS_{regiao}.parquet"
        caminho_saida = os.path.join(DIR_SAIDA, nome_saida)
        df_master.to_parquet(caminho_saida, index=False, compression='snappy')

        tamanho_mb = os.path.getsize(caminho_saida) / (1024 * 1024)
        n_linhas   = len(df_master)
        n_colunas  = len(df_master.columns)
        n_features = n_colunas - len(KEYS_MERGE)

        try:
            ano_min = int(df_master['KEY_ANO'].min())
            ano_max = int(df_master['KEY_ANO'].max())
            periodo = f"{ano_min}–{ano_max}"
        except Exception:
            periodo = "N/A"

        completude = (df_master.notna().sum().sum() /
                      (n_linhas * n_colunas) * 100)

        STATS_EXECUCAO.append({
            "Região":           regiao,
            "Linhas":           n_linhas,
            "Colunas Totais":   n_colunas,
            "Features":         n_features,
            "Arqs Unificados":  arquivos_processados,
            "Ausentes":         len(ausentes),
            "ONS":              contadores["fonte"]["ONS"],
            "INMET":            contadores["fonte"]["INMET"],
            "CCEE":             contadores["fonte"]["CCEE"],
            "BCB":              contadores["fonte"]["BCB"],
            "Feat Operacional": contadores["tipo"]["feature_operacional"],
            "Feat Meteo":       contadores["tipo"]["feature_meteorologica"],
            "Target":           contadores["tipo"]["target"],
            "Feat Financeira":  contadores["tipo"]["feature_financeira"],
            "Tamanho (MB)":     f"{tamanho_mb:.2f}",
            "Completude (%)":   f"{completude:.1f}",
            "Período":          periodo,
            "Tempo (s)":        f"{tempo_total:.2f}",
        })

        console.print(f"   [bold green]✅ Sucesso: {nome_saida}[/]")
        console.print(
            f"   📊 Shape: {df_master.shape} | "
            f"Features: {n_features} | "
            f"Completude: {completude:.1f}% | "
            f"Tamanho: {tamanho_mb:.2f} MB"
        )

        if ausentes:
            console.print(
                f"   [yellow]⚠️  Ausentes ({len(ausentes)}): "
                f"{', '.join(ausentes)}[/]"
            )
    else:
        console.print(
            f"   [bold red]❌ Falha Crítica: sem dados consolidados "
            f"para {regiao}[/]"
        )

    gc.collect()

# ==============================================================================
# 📄 GERADOR DE RELATÓRIO PDF
# ==============================================================================

def gerar_pdf():
    if not LOGS_PDF:
        return

    timestamp   = datetime.now().strftime("%d-%m-%Y_%H-%M-%S")
    arquivo_pdf = os.path.join(DIR_RELATORIO,
                               f"Relatorio_Consolidacao_{timestamp}.pdf")

    doc      = SimpleDocTemplate(arquivo_pdf, pagesize=landscape(A4))
    elements = []
    styles   = getSampleStyleSheet()

    titulo_style = ParagraphStyle(
        'titulo', parent=styles['Title'], fontSize=16, alignment=TA_CENTER)
    subtit_style = ParagraphStyle(
        'subtit', parent=styles['Normal'], fontSize=9,
        textColor=colors.grey, alignment=TA_CENTER)

    elements.append(Paragraph(
        "Relatório de Consolidação de Dados — TCC PLD", titulo_style))
    elements.append(Paragraph(
        f"Pipeline Step 3 · Gerado em: {timestamp}", subtit_style))
    elements.append(Spacer(1, 14))
    elements.append(HRFlowable(width="100%", color=colors.darkblue))
    elements.append(Spacer(1, 10))

    # ── Tabela de Resumo Estatístico ──────────────────────────────────────────
    elements.append(Paragraph("Resumo das Dimensões Finais", styles['Heading2']))
    elements.append(Spacer(1, 6))

    stat_headers = [['Região', 'Linhas', 'Features', 'Colunas\nTotais',
                     'Arqs\nUnif.', 'Ausentes', 'Compl.\n(%)', 'Período', 'MB', 'Tempo(s)']]
    stat_data = [[
        s["Região"], f"{s['Linhas']:,}", str(s["Features"]),
        str(s["Colunas Totais"]), str(s["Arqs Unificados"]),
        str(s["Ausentes"]), s["Completude (%)"], s["Período"],
        s["Tamanho (MB)"], s["Tempo (s)"]
    ] for s in STATS_EXECUCAO]

    t_stat = PDFTable(stat_headers + stat_data,
                      colWidths=[3*cm, 2.2*cm, 2.2*cm, 2.2*cm,
                                 2*cm, 2*cm, 2.2*cm, 2.5*cm, 2*cm, 2*cm])
    t_stat.setStyle(TableStyle([
        ('BACKGROUND',     (0, 0), (-1, 0), colors.darkblue),
        ('TEXTCOLOR',      (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN',          (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME',       (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE',       (0, 0), (-1, -1), 8),
        ('GRID',           (0, 0), (-1, -1), 0.4, colors.lightgrey),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1),
         [colors.white, colors.Color(0.95, 0.97, 1.0)]),
    ]))
    elements.append(t_stat)
    elements.append(Spacer(1, 16))

    # ── Tabela por Fonte / Tipo ───────────────────────────────────────────────
    elements.append(Paragraph("Distribuição por Fonte de Dados", styles['Heading2']))
    elements.append(Spacer(1, 6))

    fonte_headers = [['Região',
                      'ONS\n(feat. op.)', 'INMET\n(feat. meteo)',
                      'CCEE\n(target)', 'BCB\n(feat. fin.)',
                      'Total\nFontes']]
    fonte_data = [[
        s["Região"],
        str(s["ONS"]), str(s["INMET"]),
        str(s["CCEE"]), str(s["BCB"]),
        str(s["ONS"] + s["INMET"] + s["CCEE"] + s["BCB"])
    ] for s in STATS_EXECUCAO]

    t_fonte = PDFTable(fonte_headers + fonte_data,
                       colWidths=[3*cm, 3.5*cm, 3.5*cm, 3.5*cm, 3.5*cm, 3*cm])
    t_fonte.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0),  colors.HexColor('#003366')),
        ('TEXTCOLOR',  (0, 0), (-1, 0),  colors.whitesmoke),
        ('BACKGROUND', (1, 1), (1, -1),  colors.HexColor('#E6F1FB')),
        ('BACKGROUND', (2, 1), (2, -1),  colors.HexColor('#EAF3DE')),
        ('BACKGROUND', (3, 1), (3, -1),  colors.HexColor('#FAEEDA')),
        ('BACKGROUND', (4, 1), (4, -1),  colors.HexColor('#EEEDFE')),
        ('ALIGN',      (0, 0), (-1, -1), 'CENTER'),
        ('FONTSIZE',   (0, 0), (-1, -1), 8),
        ('GRID',       (0, 0), (-1, -1), 0.4, colors.lightgrey),
    ]))
    elements.append(t_fonte)
    elements.append(Spacer(1, 16))

    # ── Log Detalhado ─────────────────────────────────────────────────────────
    elements.append(Paragraph("Log Detalhado por Dataset", styles['Heading2']))
    elements.append(Spacer(1, 6))

    log_headers = [['Região', 'Dataset', 'Fonte', 'Tipo', 'Status', 'Observação']]
    t_log = PDFTable(log_headers + LOGS_PDF,
                     colWidths=[2.5*cm, 4.5*cm, 2*cm, 4*cm, 2.2*cm, 7.3*cm])

    log_style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
        ('TEXTCOLOR',  (0, 0), (-1, 0), colors.whitesmoke),
        ('ALIGN',      (0, 0), (-1, -1), 'LEFT'),
        ('FONTSIZE',   (0, 0), (-1, -1), 7),
        ('GRID',       (0, 0), (-1, -1), 0.3, colors.lightgrey),
    ])
    for i, row in enumerate(LOGS_PDF):
        status = row[4]
        if status in ["ERRO", "AUSENTE"]:
            log_style.add('TEXTCOLOR', (0, i+1), (-1, i+1), colors.red)
        elif status in ["BASE", "MERGED"]:
            log_style.add('TEXTCOLOR', (4, i+1), (4, i+1),
                          colors.HexColor('#1D7A46'))

    t_log.setStyle(log_style)
    elements.append(t_log)

    try:
        doc.build(elements)
        console.print(f"\n[bold white]📄 PDF Gerado: {arquivo_pdf}[/]")
    except Exception as e:
        console.print(f"[red]Erro ao gerar PDF: {e}[/]")

# ==============================================================================
# 🚀 EXECUÇÃO PRINCIPAL
# ==============================================================================

def main():
    console.print(Panel.fit(
        "[bold white]🚀 CONSOLIDAÇÃO DE DADOS (STEP 3 — v3.5)[/]",
        style="header", box=box.ROUNDED
    ))

    os.makedirs(DIR_SAIDA,     exist_ok=True)
    os.makedirs(DIR_RELATORIO, exist_ok=True)

    for reg in REGIOES:
        # 1. Consolida normalmente
        processar_regiao(reg)

        # 2. Diagnóstico automático do período alvo após cada região
        diagnosticar_maio(reg)

    # ── Tabela resumo no terminal ──────────────────────────────────────────────
    console.print("\n[bold white on blue] 📊 RESUMO DA EXECUÇÃO [/]")

    tabela = RichTable(
        title="Estatísticas da Consolidação", box=box.SIMPLE_HEAVY)
    for col in ["Região", "Linhas", "Features", "Colunas", "Arqs", "Ausentes",
                "ONS", "INMET", "CCEE", "BCB", "Compl.(%)", "Período", "MB", "Tempo"]:
        tabela.add_column(
            col, justify="right" if col not in ["Região", "Período"] else "left")

    for s in STATS_EXECUCAO:
        tabela.add_row(
            s["Região"],
            f"{s['Linhas']:,}",
            str(s["Features"]),
            str(s["Colunas Totais"]),
            str(s["Arqs Unificados"]),
            str(s["Ausentes"]),
            str(s["ONS"]),
            str(s["INMET"]),
            str(s["CCEE"]),
            str(s["BCB"]),
            s["Completude (%)"],
            s["Período"],
            f"{s['Tamanho (MB)']} MB",
            f"{s['Tempo (s)']}s",
        )

    console.print(tabela)

    # ── Aviso específico sobre Norte ──────────────────────────────────────────
    norte = next((s for s in STATS_EXECUCAO if s["Região"] == "NORTE"), None)
    if norte and norte["Ausentes"] > 0:
        console.print(Panel(
            f"[yellow]⚠️  O submercado NORTE possui {norte['Ausentes']} dataset(s) "
            f"ausente(s), resultando em {norte['Features']} features contra "
            f"~{max(s['Features'] for s in STATS_EXECUCAO)} do Sudeste.\n"
            f"Isso pode impactar modelos treinados especificamente nessa região.\n"
            f"Recomendação: verificar disponibilidade dos CSVs de origem no ETL.\n"
            f"Dataset sem arquivo para NORTE: {sorted(DATASETS_AUSENTES_NORTE)}[/]",
            title="Análise: Submercado Norte",
            border_style="yellow"
        ))

    gerar_pdf()
    console.print("\n[bold green]🏁 Processo Finalizado com Sucesso![/]")


if __name__ == "__main__":
    main()